# <font color='orange'>**RapidRelief AI — Sprint 1: EDA y Preprocesamiento**</font>

**Objetivo:** Analizar la distribución del Clothing Dataset Small y preparar los  
generadores de entrenamiento, validación y test para MobileNetV2 (Transferencia de Aprendizaje).

## 1. Configuración inicial

In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
%matplotlib inline

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Rutas del proyecto
BASE_DIR    = '/content/drive/MyDrive/RapidReliefAI'
CLOTH_TRAIN = BASE_DIR + '/Datasets/clothing_small/train'
CLOTH_VAL   = BASE_DIR + '/Datasets/clothing_small/validation'
CLOTH_TEST  = BASE_DIR + '/Datasets/clothing_small/test'

# Parametros de imagen — igual que en clase
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

# Clases finales del proyecto (orden alfabetico, como flow_from_directory las asigna)
# 0=dress 1=hat 2=longsleeve 3=outwear 4=pants 5=shirt 6=shoes 7=shorts 8=skirt 9=t-shirt
CLASES = ['dress', 'hat', 'longsleeve', 'outwear', 'pants',
          'shirt', 'shoes', 'shorts', 'skirt', 't-shirt']
N_CLASES = len(CLASES)
print(f'Numero de clases: {N_CLASES}')
print(f'Clases: {CLASES}')

## 2. EDA — Clothing Dataset Small

In [ ]:
# Conteo de imagenes por clase en cada split
def contar_por_clase(directorio):
    conteo = {}
    for clase in sorted(os.listdir(directorio)):
        ruta = os.path.join(directorio, clase)
        if os.path.isdir(ruta):
            n = len([f for f in os.listdir(ruta)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
            conteo[clase] = n
    return conteo

conteo_train = contar_por_clase(CLOTH_TRAIN)
conteo_val   = contar_por_clase(CLOTH_VAL)
conteo_test  = contar_por_clase(CLOTH_TEST)

df_conteo = pd.DataFrame({
    'train': conteo_train,
    'val':   conteo_val,
    'test':  conteo_test
})
df_conteo['total'] = df_conteo.sum(axis=1)
print('Clothing Dataset Small — Distribucion de clases:')
print(df_conteo)
print(f'\nTotales: train={df_conteo["train"].sum()} | val={df_conteo["val"].sum()} | test={df_conteo["test"].sum()}')

In [ ]:
# Grafica de distribucion — identificar clases desbalanceadas
fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(df_conteo))
width = 0.25

ax.bar(x - width, df_conteo['train'], width, label='Train', color='steelblue')
ax.bar(x,         df_conteo['val'],   width, label='Val',   color='orange')
ax.bar(x + width, df_conteo['test'],  width, label='Test',  color='green')

ax.set_xticks(x)
ax.set_xticklabels(df_conteo.index, rotation=30, ha='right')
ax.set_title('Distribucion de clases — Clothing Dataset Small')
ax.set_ylabel('Num. imagenes')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Visualizar una imagen de cada clase
from tensorflow.keras.preprocessing.image import load_img, img_to_array

def plotImages(images_arr, titulos=None):
    n = len(images_arr)
    fig, axes = plt.subplots(1, n, figsize=(n * 2, 3))
    if n == 1:
        axes = [axes]
    for i, (img, ax) in enumerate(zip(images_arr, axes)):
        ax.imshow(img.astype('uint8'))
        ax.axis('off')
        if titulos:
            ax.set_title(titulos[i], fontsize=8)
    plt.tight_layout()
    plt.show()

imagenes = []
titulos = []
for i, clase in enumerate(sorted(os.listdir(CLOTH_TRAIN))):
    ruta_clase = os.path.join(CLOTH_TRAIN, clase)
    if os.path.isdir(ruta_clase):
        archivos = [f for f in os.listdir(ruta_clase)
                    if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        if archivos:
            img = load_img(os.path.join(ruta_clase, archivos[0]), target_size=IMAGE_SIZE)
            imagenes.append(img_to_array(img))
            titulos.append(f'{i}: {clase}')

print('Muestras del Clothing Dataset Small (una por clase):')
plotImages(imagenes, titulos)
# 0=dress 1=hat 2=longsleeve 3=outwear 4=pants 5=shirt 6=shoes 7=shorts 8=skirt 9=t-shirt

## 3. Generadores de datos — Clothing Small

Siguiendo la metodología de clase: `ImageDataGenerator` + `flow_from_directory`  
con `preprocessing_function=preprocess_input` y augmentación robusta para
compensar el tamaño moderado del dataset (~3,068 imágenes de entrenamiento).

In [ ]:
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Generador de entrenamiento con augmentation robusta
# Sin Fashion-MNIST como complemento, se aumenta la diversidad sintética
# del único dataset (Clothing Small) mediante augmentation.
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=15,
    width_shift_range=0.10,
    height_shift_range=0.10,
    shear_range=10.0,
    zoom_range=0.15,
    horizontal_flip=True,
    brightness_range=[0.85, 1.15],
    fill_mode='nearest'
)

# Generador de validacion y test — sin augmentation
val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

print('Cargando datos de entrenamiento (Clothing Small)...')
train_data = train_datagen.flow_from_directory(
    CLOTH_TRAIN,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    seed=42
)

print('\nCargando datos de validacion (Clothing Small)...')
val_data = val_datagen.flow_from_directory(
    CLOTH_VAL,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    seed=42
)

print('\nCargando datos de test (Clothing Small)...')
test_data = val_datagen.flow_from_directory(
    CLOTH_TEST,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    seed=42
)

print('\nIndice de clases (asignado por flow_from_directory):')
print(train_data.class_indices)
# Confirmar: {'dress':0, 'hat':1, 'longsleeve':2, 'outwear':3, 'pants':4,
#             'shirt':5, 'shoes':6, 'shorts':7, 'skirt':8, 't-shirt':9}

In [ ]:
# Verificar forma de un batch
imgs, labels = next(train_data)
print(f'Forma del batch de imagenes: {imgs.shape}')   # (32, 224, 224, 3)
print(f'Forma del batch de etiquetas: {labels.shape}') # (32, 10) — One-Hot Encoding
print(f'Rango de valores de imagen: [{imgs.min():.4f}, {imgs.max():.4f}]')  # [-1, 1]

In [ ]:
# Visualizar un batch con augmentation aplicado
# Nota: imshow espera [0,255], revertimos preprocess_input para visualizacion
imgs_vis = (imgs + 1.0) / 2.0  # de [-1,1] a [0,1]
clases_invertidas = {v: k for k, v in train_data.class_indices.items()}
titulos_batch = [clases_invertidas[np.argmax(l)] for l in labels[:10]]

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
axes = axes.flatten()
for i in range(10):
    axes[i].imshow(imgs_vis[i])
    axes[i].set_title(titulos_batch[i], fontsize=9)
    axes[i].axis('off')
plt.suptitle('Batch de entrenamiento con augmentation aplicada', fontsize=11)
plt.tight_layout()
plt.show()

## 4. Resumen del Sprint 1

Los generadores están listos. Continuar con `02_transfer_learning.ipynb`.

In [ ]:
print('=== RESUMEN SPRINT 1 ===')
print(f'Clases finales ({N_CLASES}): {CLASES}')
print(f'Clothing Small — train: {train_data.samples} | val: {val_data.samples} | test: {test_data.samples}')
print(f'IMAGE_SIZE: {IMAGE_SIZE}  |  BATCH_SIZE: {BATCH_SIZE}')
print(f'Augmentation: rotation=15, shear=10, zoom=0.15, h_flip, brightness=[0.85,1.15], shifts=0.10')
print(f'Preprocesamiento: MobileNetV2 preprocess_input → rango [-1, 1]')
print('Generadores verificados. Listo para entrenamiento.')